# Trading patterns 本地说明

## 目标
- 从 `returns.parquet` 和 `screen_aggregate.parquet` 生成 `patterns.parquet`
- 结果表保存在 `patterns_total`，便于检查中间结果

## 运行环境
- 仅支持 Python 3.10 到 3.13，推荐 Python 3.12
- 当前项目依赖 `pandas-ta` 和 `numba`，Python 3.14 无法安装完整依赖

## 输入数据
- `returns.parquet`：日频收益宽表
- `screen_aggregate.parquet`：月末截面宽表
- 字段和连接关系参考 `screen_returns_context.md`

## 推荐执行方式
- 想理解流程：按顺序执行下方“主流程”单元
- 想快速跑通：只运行下方“一键运行”单元
- 两条路径二选一，不要混跑

## 输出验证
- 优先看 `patterns_total.shape` 和 `patterns_total.head()`
- notebook 中已有输出可能是历史缓存，请以重新运行后的结果为准

In [10]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import pyarrow
import pandas as pd
from utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import os
from pathlib import Path
# 默认读取 TP canonical 数据；如需复现实验快照，可设置 TA_RETURNS_PATH / TA_SCREEN_PATH / TA_OUTPUT_PATH
project_dir = Path.cwd()
output_dir = project_dir / "output"

from tp_core.data_sources import RETURNS_PATH as CANONICAL_RETURNS_PATH
from tp_core.data_sources import SCREEN_AGGREGATE_PATH

path_returns = Path(os.environ.get("TA_RETURNS_PATH", str(CANONICAL_RETURNS_PATH)))
path_screen = Path(os.environ.get("TA_SCREEN_PATH", str(SCREEN_AGGREGATE_PATH)))
path_output_pattern = Path(os.environ.get("TA_OUTPUT_PATH", str(output_dir / "patterns.parquet")))

output_dir.mkdir(parents=True, exist_ok=True)

if not path_returns.exists() or not path_screen.exists():
    raise FileNotFoundError(
        "未找到输入 parquet。请确认 TP canonical 数据存在，或通过 TA_RETURNS_PATH / TA_SCREEN_PATH 指定实验快照路径。"
    )

returns = pd.read_parquet(path_returns)
screen_agg = pd.read_parquet(path_screen)


In [3]:
filter_univ = (screen_agg["Weight in SP500"]>0) | (screen_agg["Weight in STOXX EUROPE 600"]>0)

In [4]:
isin_to_compute = screen_agg[filter_univ]['Company SEDOL'].dropna().unique()
valid_cols = [
    col for col in isin_to_compute
    if col and str(col).lower() != "none" and col in returns.columns
]

print(f"可计算证券数: {len(valid_cols)} / 原始候选数: {len(isin_to_compute)}")
if not valid_cols:
    raise ValueError("未找到可用于计算的证券列，请检查 screen_aggregate.parquet 与 returns.parquet 的 Company SEDOL 是否一致。")

可计算证券数: 1977 / 原始候选数: 1978


In [5]:
# 可选：如果只想测试少量证券，可以在这里手工覆盖 valid_cols
# valid_cols = ["ABCDE1-R", "FGHIJ2-R"]

## 主流程（按顺序执行）

默认使用全量历史数据，不设置 `start_date`。
如果你只是想本地快速跑通，请直接跳到下方“一键运行”部分；不要和这里的分步单元混跑。

In [6]:
patterns = detect_pattern_(returns[valid_cols])
patterns = patterns.stack(level=0)

c:\GoogleDrive\TP\技术分析和深度学习\技术分析_V2\utils.py:91: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
c:\GoogleDrive\TP\技术分析和深度学习\技术分析_V2\utils.py:91: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
C:\Users\jingx\AppData\Local\Temp\ipykernel_84244\2434961717.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  patterns = patterns.stack(level=0)


In [7]:

patterns_candle = detect_pattern_(returns[valid_cols], library='candlestick')
patterns_candle = patterns_candle.loc[:, ~patterns_candle.columns.get_level_values(1).isin(['low', 'open', 'close', 'high'])].stack(level=0)
patterns_inter = pd.concat([patterns, patterns_candle.loc[patterns.index]], axis=1)



c:\GoogleDrive\TP\技术分析和深度学习\技术分析_V2\utils.py:91: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
c:\GoogleDrive\TP\技术分析和深度学习\技术分析_V2\utils.py:91: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
C:\Users\jingx\AppData\Local\Temp\ipykernel_84244\3181105291.py:2: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  patterns_candle = patterns_candle.loc[:, ~patterns_candle.columns.get_level_values(1).isin(['low', 'open', 'close', 'high'])].stack

In [8]:
patterns_inter.index = patterns_inter.index.reorder_levels([1,0])
patterns_inter.index.names = ['Company SEDOL', 'Date']

In [11]:

# calcul des indicateurs supplémentaires grace a pandas ta 
indicator=patterns_inter[['Open','High','Low','Close']].groupby(level=0,axis=0).apply(calcul_indicator).droplevel(0)
patterns_total = pd.concat([patterns_inter,indicator.iloc[:,4:]],axis=1)

C:\Users\jingx\AppData\Local\Temp\ipykernel_84244\1770945744.py:2: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  indicator=patterns_inter[['Open','High','Low','Close']].groupby(level=0,axis=0).apply(calcul_indicator).droplevel(0)
C:\Users\jingx\AppData\Local\Temp\ipykernel_84244\1770945744.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  indicator=patterns_inter[['Open','High','Low','Close']].groupby(level=0,axis=0).apply(calcul_indicator).droplevel(0)


In [12]:
patterns_total = patterns_total.reset_index()
patterns_total.set_index("Company SEDOL", inplace=True, drop=True)

print(patterns_total.shape)
patterns_total.head()

(2183748, 98)


,Date,Low,High,Close,Open,high_roll_max,low_roll_min,trend_high,trend_low,close_roll_max,...,BBM_10_2.0_2.0_10_1.5,BBU_10_2.0_2.0_10_1.5,BBB_10_2.0_2.0_10_1.5,BBP_10_2.0_2.0_10_1.5,atr_14,atr_21,atr_30,stdev_10,stdev_20,stdev_30
Company SEDOL,,,,,,,,,,,,,,,,,,,,,
B02662-R,2005-01-03,98.461538,102.991455,102.991455,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B045ZY-R,2005-01-03,99.438512,101.057654,101.057654,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B0D4Y5-R,2005-01-03,100.000000,100.000000,100.000000,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B18MVG-R,2005-01-03,98.940350,100.013359,100.013359,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
B19ST9-R,2005-01-03,99.917388,103.187467,103.187467,100.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
patterns_total.to_parquet(path_output_pattern)

## 一键运行（可选）

如果你只想本地快速跑通，可以只运行下面这个单元。
如果已经执行了上面的分步主流程，请不要再运行这一段。

In [ ]:
import os
from pathlib import Path
import pyarrow
from utils import *
import pandas as pd

# 默认读取 TP canonical 数据；如需复现实验快照，可设置 TA_RETURNS_PATH / TA_SCREEN_PATH / TA_OUTPUT_PATH
project_dir = Path.cwd()
output_dir = project_dir / "output"

from tp_core.data_sources import RETURNS_PATH as CANONICAL_RETURNS_PATH
from tp_core.data_sources import SCREEN_AGGREGATE_PATH

path_returns = Path(os.environ.get("TA_RETURNS_PATH", str(CANONICAL_RETURNS_PATH)))
path_screen = Path(os.environ.get("TA_SCREEN_PATH", str(SCREEN_AGGREGATE_PATH)))
path_output_pattern = Path(os.environ.get("TA_OUTPUT_PATH", str(output_dir / "patterns.parquet")))

output_dir.mkdir(parents=True, exist_ok=True)

if not path_returns.exists() or not path_screen.exists():
    raise FileNotFoundError(
        "未找到输入 parquet。请确认 TP canonical 数据存在，或通过 TA_RETURNS_PATH / TA_SCREEN_PATH 指定实验快照路径。"
    )

returns = pd.read_parquet(path_returns)
screen_agg = pd.read_parquet(path_screen)

filter_univ = (screen_agg["Weight in SP500"] > 0) | (screen_agg["Weight in STOXX EUROPE 600"] > 0)
isin_to_compute = screen_agg[filter_univ]["Company SEDOL"].dropna().unique()
valid_cols = [
    col for col in isin_to_compute
    if col and str(col).lower() != "none" and col in returns.columns
]

print(f"可计算证券数: {len(valid_cols)} / 原始候选数: {len(isin_to_compute)}")
if not valid_cols:
    raise ValueError("未找到可用于计算的证券列，请检查 screen_aggregate.parquet 与 returns.parquet 的 Company SEDOL 是否一致。")

patterns = detect_pattern_(returns[valid_cols])
patterns = patterns.stack(level=0)

patterns_candle = detect_pattern_(returns[valid_cols], library="candlestick")
patterns_candle = patterns_candle.loc[:, ~patterns_candle.columns.get_level_values(1).isin(["low", "open", "close", "high"])].stack(level=0)
patterns_inter = pd.concat([patterns, patterns_candle.loc[patterns.index]], axis=1)

patterns_inter.index = patterns_inter.index.reorder_levels([1, 0])
patterns_inter.index.names = ["Company SEDOL", "Date"]

indicator = patterns_inter[["Open", "High", "Low", "Close"]].groupby(level=0, axis=0).apply(calcul_indicator).droplevel(0)
patterns_total = pd.concat([patterns_inter, indicator.iloc[:, 4:]], axis=1)

patterns_total = patterns_total.reset_index()
patterns_total.set_index("Company SEDOL", inplace=True, drop=True)
patterns_total.to_parquet(path_output_pattern)

print(patterns_total.shape)
patterns_total.head()


\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\DOSSIERS_UTILISATEURS\Yohan\Git\Analyse_Technique\utils.py:84: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\DOSSIERS_UTILISATEURS\Yohan\Git\Analyse_Technique\utils.py:84: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  df_grouped=df.set_index((index_name,'Low')).drop(index_name,axis=1).rename_axis('Date').groupby(level=0,axis=1)
C:\Users\RADETYO\AppData\Local\Temp\ipykernel_10616\239212288.py:18: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and s

In [15]:
# 历史重复单元，已并入上方流程，无需重复执行

In [20]:
# 历史重复单元，已并入上方流程，无需重复执行

In [21]:
# 历史调试单元，已停用

,Company SEDOL,Date,Low,High,Close,Open,high_roll_max,low_roll_min,trend_high,trend_low,...,Doji,Hammer,HangingMan,InvertedHammer,ShootingStar,MorningStar,BullishHarami,BearishHarami,PiercingPattern,DarkCloudCover
0,B01DPB-R,2003-01-02,94.495082,100.000000,94.495082,100.000000,NaN,NaN,NaN,NaN,...,False,False,None,False,None,None,None,None,None,None
1,B02662-R,2003-01-02,100.000000,100.000000,100.000000,100.000000,NaN,NaN,NaN,NaN,...,False,False,None,False,None,None,None,None,None,None
2,B045ZY-R,2003-01-02,100.000000,101.256400,101.256400,100.000000,NaN,NaN,NaN,NaN,...,False,False,None,False,None,None,None,None,None,None
3,B0TXKG-R,2003-01-02,100.000000,100.000000,100.000000,100.000000,NaN,NaN,NaN,NaN,...,False,False,None,False,None,None,None,None,None,None
4,B18MVG-R,2003-01-02,99.935400,100.000000,99.935400,100.000000,NaN,NaN,NaN,NaN,...,False,False,None,False,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3267125,XQ818C-R,2025-11-17,477.973389,477.973389,477.973389,477.973389,492.504870,474.864689,-1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
3267126,XQCN4R-R,2025-11-17,262.887918,262.887918,262.887918,262.887918,268.228175,236.704402,1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
3267127,XQDGQJ-R,2025-11-17,57.318571,57.318571,57.318571,57.318571,59.113948,57.265433,-1.0,-1.0,...,False,False,False,False,False,False,False,False,False,False
3267128,XQF8XY-R,2025-11-17,236.783735,236.783735,236.783735,236.783735,240.775372,235.219649,-1.0,1.0,...,False,False,False,False,False,False,False,False,False,False
